In [1]:
import os
os.makedirs("/workspace/98359772-f251-4284-9fcb-d126816adbeb", exist_ok=True)
print("Directory created.")


Directory created.


In [2]:
# Let's first try to fetch zeros from LMFDB via their API
# The L-function label is 2-1-1.1-c11-0-0
import httpx
import asyncio

async def try_lmfdb():
 urls = [
 "https://www.lmfdb.org/api/lfunc_zeros/?label=2-1-1.1-c11-0-0&_format=json",
 "https://www.lmfdb.org/api/lfunc_lfunctions/?label=2-1-1.1-c11-0-0&_format=json",
 "https://www.lmfdb.org/L/2/1/1.1/c11/0/0",
 ]
 async with httpx.AsyncClient(timeout=30, follow_redirects=True) as client:
 for url in urls[:2]:
 try:
 resp = await client.get(url)
 print(f"URL: {url}")
 print(f"Status: {resp.status_code}")
 print(f"Content (first 2000): {resp.text[:2000]}")
 print()
 except Exception as e:
 print(f"URL: {url} -> Error: {e}")
 print()

await try_lmfdb()


URL: https://www.lmfdb.org/api/lfunc_zeros/?label=2-1-1.1-c11-0-0&_format=json
Status: 404
Content (first 2000): <!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - LMFDB Page Not Found </title>
 <link id="style_css" href="/style.css" rel="stylesheet" type="text/css" /> 

 <!-- jQuery -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"
 integrity="sha384-xBuQ/xzmlsLoJpyjoggmTEz8OWUFM0/RC5BsqQBDX2v5cMvDHcMakNTNrHIW2I5f"
 crossorigin="anonymous"></script>
 <!-- jQuery UI -->
 <script type="text/javascript"
 src="https://ajax.googlea

In [3]:
# Good, we got the LMFDB entry. Let's see if there are zeros included
import json

async def get_lmfdb_data():
 async with httpx.AsyncClient(timeout=30, follow_redirects=True) as client:
 resp = await client.get(
 "https://www.lmfdb.org/api/lfunc_lfunctions/",
 params={"label": "2-1-1.1-c11-0-0", "_format": "json"}
 )
 data = resp.json()["data"][0]
 # Check if zeros are included
 relevant_keys = [k for k in data.keys() if 'zero' in k.lower() or 'z' in k.lower()]
 print("Keys with 'zero' or 'z':", relevant_keys)
 
 # Let's see all keys
 print("\nAll keys:", list(data.keys()))
 
 # Check for zeros specifically
 for key in ['positive_zeros', 'z1', 'z2', 'z3', 'zeros', 'first_zeros']:
 if key in data:
 print(f"\n{key}: {data[key]}")
 
 # Check 'positive_zeros' or similar
 if 'positive_zeros' in data:
 print(f"\npositive_zeros: {data['positive_zeros'][:5]}")
 
 return data

data = await get_lmfdb_data()


Keys with 'zero' or 'z': ['analytic_normalization', 'euler_factors_factorization', 'positive_zeros', 'z1', 'z2', 'z3']

All keys: ['id', 'Lhash', 'a10', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'accuracy', 'algebraic', 'analytic_conductor', 'analytic_normalization', 'bad_lfactors', 'bad_primes', 'central_character', 'coefficient_field', 'conductor', 'conductor_radical', 'degree', 'euler_factors', 'euler_factors_factorization', 'gamma_factors', 'group', 'index', 'label', 'load_key', 'motivic_weight', 'mu_imag', 'mu_real', 'nu_imag', 'nu_real_doubled', 'order_of_vanishing', 'origin', 'plot_delta', 'plot_values', 'positive_zeros', 'prelabel', 'primitive', 'rational', 'root_analytic_conductor', 'root_angle', 'root_number', 'self_dual', 'sign_arg', 'spectral_label', 'trace_hash', 'types', 'z1', 'z2', 'z3']

positive_zeros: ['9.222379399921102522243767192743263', '13.90754986139213440644668132877006', '17.44277697823447331355152513712760', '19.65651314195496100012728175632157', '22.33

In [4]:
# Great! LMFDB has positive_zeros. Let's check how many they provide
zeros = data['positive_zeros']
print(f"Number of zeros from LMFDB: {len(zeros)}")
print(f"First zero: {zeros[0]}")
print(f"Last zero: {zeros[-1]}")
for i, z in enumerate(zeros):
 print(f" γ_{i+1} = {z}")


Number of zeros from LMFDB: 10
First zero: 9.222379399921102522243767192743263
Last zero: 32.77487538223120744183045567331179
 γ_1 = 9.222379399921102522243767192743263
 γ_2 = 13.90754986139213440644668132877006
 γ_3 = 17.44277697823447331355152513712760
 γ_4 = 19.65651314195496100012728175632157
 γ_5 = 22.33610363720986727568267445923646
 γ_6 = 25.27463654811236535674532419313339
 γ_7 = 26.80439115835040303257574923358447
 γ_8 = 28.83168262418687544502196191298445
 γ_9 = 31.17820949836025906449218889077411
 γ_10 = 32.77487538223120744183045567331179


In [5]:
# Only 10 zeros from LMFDB. We need 80. Let's try their download/zeros endpoint
# or see if there's a way to get more zeros

async def try_more_zeros():
 async with httpx.AsyncClient(timeout=30, follow_redirects=True) as client:
 # Try the zeros endpoint directly
 urls_to_try = [
 "https://www.lmfdb.org/api/lfunc_zeros/?Lhash=11690754781842443695803342913187&_format=json",
 "https://www.lmfdb.org/L/2/1/1.1/c11/0/0/zeros",
 "https://www.lmfdb.org/api/lfunc_lfunctions/?label=2-1-1.1-c11-0-0&_fields=positive_zeros&_format=json&_max_count=100",
 ]
 for url in urls_to_try:
 try:
 resp = await client.get(url)
 print(f"URL: {url}")
 print(f"Status: {resp.status_code}")
 content = resp.text[:1500]
 print(f"Content: {content}")
 print()
 except Exception as e:
 print(f"URL: {url} -> Error: {e}\n")

await try_more_zeros()


URL: https://www.lmfdb.org/api/lfunc_zeros/?Lhash=11690754781842443695803342913187&_format=json
Status: 404
Content: <!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - LMFDB Page Not Found </title>
 <link id="style_css" href="/style.css" rel="stylesheet" type="text/css" /> 

 <!-- jQuery -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"
 integrity="sha384-xBuQ/xzmlsLoJpyjoggmTEz8OWUFM0/RC5BsqQBDX2v5cMvDHcMakNTNrHIW2I5f"
 crossorigin="anonymous"></script>
 <!-- jQuery UI -->
 <script type="text/javascript"
 src="https://ajax.goo

In [6]:
import subprocess
subprocess.run(["pip", "install", "mpmath"], capture_output=True, text=True)
import mpmath
print(f"mpmath version: {mpmath.__version__}")


mpmath version: 1.4.1


In [7]:
# Now let's build the L-function computation.
# 
# The Ramanujan Delta L-function in ANALYTIC normalization:
# L(s) = sum_{n=1}^{infty} a(n) / n^s where a(n) = tau(n)/n^{11/2}
# so that the critical line is Re(s) = 1/2.
#
# Equivalently, in arithmetic normalization L_arith(s) = sum tau(n)/n^s
# and zeros are at s = 6 + i*gamma. We want gamma values.
#
# The completed L-function:
# Lambda(s) = (2*pi)^{-s} * Gamma(s + 11/2) * L_arith(s + 11/2) ... hmm, 
# let me be more careful.
#
# For the modular form Delta of weight 12, level 1:
# L_arith(s) = sum_{n>=1} tau(n) * n^{-s}, converging for Re(s) > 13/2
# The completed L-function: Lambda(s) = (2*pi)^{-s} * Gamma(s) * L_arith(s)
# Functional equation: Lambda(s) = Lambda(12-s) [since weight 12, level 1]
# Center of symmetry: s = 6
#
# In analytic normalization: L_an(s) = L_arith(s + 11/2) 
# so zeros at 1/2 + i*gamma correspond to arithmetic zeros at 6 + i*gamma
# gamma values are the same in both normalizations!
#
# The Hardy Z-function approach: 
# Z(t) = e^{i*theta(t)} * L(1/2 + i*t)
# where theta is chosen so Z(t) is real for real t.
# Zeros of Z(t) = zeros of L on the critical line.
#
# For computation, we use the approximate functional equation (Dokchitser's method)
# or direct summation with smoothing.

# Step 1: Compute Ramanujan tau coefficients
mpmath.mp.dps = 60

def compute_tau_coefficients(N):
 """Compute tau(1), ..., tau(N) using the q-expansion of Delta."""
 # Delta(q) = q * prod_{n=1}^{inf} (1 - q^n)^24
 # We compute the q-expansion coefficients up to q^N
 # Using the Dedekind eta function: Delta = eta(z)^24
 # 
 # More efficient: use recurrence for tau via Hecke operators
 # or just compute the product directly for moderate N.
 
 # For N up to a few thousand, direct computation of product is fine.
 # tau[n] = coefficient of q^n in q * prod_{k=1}^{N} (1-q^k)^24
 
 # We work with polynomial coefficients
 # Start with [1] and multiply by (1 - q^k)^24 for k=1,...,N
 
 # Actually, it's more efficient to compute (1-q^k)^24 step by step
 # Let's use the fact that prod (1-q^k)^24 and multiply by q
 
 coeffs = [mpmath.mpf(0)] * (N + 1)
 coeffs[0] = mpmath.mpf(1)
 
 for k in range(1, N + 1):
 # Multiply by (1 - q^k)^24
 # First compute (1 - q^k)^24 contributions
 # We can expand (1-x)^24 = sum_{j=0}^{24} C(24,j)*(-1)^j * x^j
 # where x = q^k
 # So multiply coeffs by sum_{j=0}^{24} C(24,j)*(-1)^j * q^{jk}
 
 from mpmath import binomial
 
 # We need to apply the multiplication in-place going backwards
 # to avoid overwriting coefficients we still need
 new_coeffs = [mpmath.mpf(0)] * (N + 1)
 for j in range(min(24, N // k) + 1):
 sign = (-1)**j
 binom = int(binomial(24, j))
 factor = sign * binom
 shift = j * k
 for n in range(N + 1 - shift):
 new_coeffs[n + shift] += factor * coeffs[n]
 coeffs = new_coeffs
 
 if k % 200 == 0:
 print(f" k={k}/{N}")
 
 # Now coeffs[n] = coefficient of q^n in prod(1-q^k)^24
 # Delta has tau(n) as coefficient of q^n, so tau(n) = coeffs[n-1] 
 # (since Delta = q * prod = sum tau(n) q^n, so tau(n) = coeffs[n-1])
 # Wait: coeffs[0] = 1 (constant term of product), and Delta = q * product
 # So coefficient of q^n in Delta = coeffs[n-1]
 # tau(1) = coeffs[0] = 1, tau(2) = coeffs[1] = -24, etc.
 
 tau = [None] + [int(coeffs[n-1]) for n in range(1, N+1)]
 return tau

print("Computing Ramanujan tau coefficients...")
N_coeffs = 800
tau = compute_tau_coefficients(N_coeffs)
print(f"Computed {N_coeffs} tau coefficients")
print(f"tau(1) = {tau[1]}")
print(f"tau(2) = {tau[2]}")
print(f"tau(3) = {tau[3]}")
print(f"tau(4) = {tau[4]}")
print(f"tau(5) = {tau[5]}")
# Known: tau(1)=1, tau(2)=-24, tau(3)=252, tau(4)=-1472, tau(5)=4830


Computing Ramanujan tau coefficients...


 k=200/800


 k=400/800


 k=600/800


 k=800/800
Computed 800 tau coefficients
tau(1) = 1
tau(2) = -24
tau(3) = 252
tau(4) = -1472
tau(5) = 4830


In [8]:
# Good! tau coefficients are correct. Now let's implement the L-function evaluation.
# 
# We'll use the approximate functional equation for L(Delta, s).
# The completed L-function:
# Lambda(s) = (2*pi)^{-s} * Gamma(s) * L_arith(s)
# where L_arith(s) = sum_{n>=1} tau(n) * n^{-s}
# Functional equation: Lambda(s) = Lambda(12 - s) [sign = +1 for Delta]
#
# The approximate functional equation:
# L_arith(s) = sum_{n<=x} tau(n)/n^s * V_s(n/sqrt(Q)) + epsilon * (stuff) * sum_{n<=y} tau(n)/n^{12-s} * V_{12-s}(n/sqrt(Q))
# where Q = (2*pi)^{-2} * 1 = 1/(4*pi^2) ... this is getting complex.
#
# Simpler approach: Use the Lavrik / Dokchitser method.
# For a self-dual L-function with sign +1:
# Lambda(s) + Lambda(12-s) = 2*Lambda(s) [since Lambda(s)=Lambda(12-s)]
# 
# Actually let me use a more direct approach. For the critical line s = 6 + it:
# We compute L_arith(6+it) using exponential smoothing via the incomplete gamma function.
#
# The key identity (approximate functional equation):
# Lambda(s) = sum_{n=1}^{inf} tau(n) * [Gamma_inc_upper(s, 2*pi*n) / (2*pi*n)^{-s} ... ]
#
# Let me use a cleaner formulation. The Mellin-Barnes approach:
# Lambda(s) = int_0^{inf} (Delta(iy) - 0) * y^{s/2} * dy/y ... no, this isn't quite right for 
# modular forms.
#
# Actually, let's use the standard integral representation:
# Lambda(s) = (2*pi)^{-s} * Gamma(s) * L(s) = integral_1^{inf} f(iy) * y^s * dy/y + 
# integral_1^{inf} f(i/y) * y^{12-s} * dy/y ... 
# but f(iy) = Delta(iy) * y^6 which decays exponentially.
#
# More concretely, using the Hecke trick for level 1:
# Lambda(s) = sum_{n=1}^{inf} tau(n) * [G(s, 2*pi*n) + G(12-s, 2*pi*n)]
# where G(s, x) = integral_1^{inf} e^{-xt} * t^{s-1} dt = x^{-s} * Gamma(s, x) [upper incomplete gamma]
# Wait, that's Gamma_upper(s, x) / x^s... 
# Actually G(s, x) = integral_1^{inf} e^{-xt} * t^{s-1} dt
#
# Hmm, let me just use a well-known, correct formula.
# 
# For the Ramanujan L-function:
# (2π)^{-s} Γ(s) L(s) = Σ_{n≥1} τ(n) [ (2πn)^{-s} Γ(s,2πn) + (2πn)^{-(12-s)} Γ(12-s,2πn) ]
#
# Actually the classic formula is:
# Λ(s) = Σ_{n=1}^∞ τ(n)/n^s · ψ(s, 2πn) + Σ_{n=1}^∞ τ(n)/n^{12-s} · ψ(12-s, 2πn)
# where ψ(s, x) = (1/(2πi)) ∫ Γ(s+w)/Γ(s) · x^{-w} dw/(w) = incomplete gamma ratio
#
# The simplest correct approach: just evaluate the rapidly convergent series
# Λ(s) = ∫_1^∞ Δ(iy) y^{s-1} dy + ∫_1^∞ Δ(iy) y^{11-s} dy
# where Δ(iy) = Σ τ(n) e^{-2πny} (note Δ(z) = Σ τ(n) e^{2πinz} = Σ τ(n) q^n with q=e^{2πiz})
# For z = iy, q = e^{-2πy}.
# 
# So:
# Λ(s) = Σ_{n≥1} τ(n) [ ∫_1^∞ e^{-2πny} y^{s-1} dy + ∫_1^∞ e^{-2πny} y^{11-s} dy ]
# = Σ_{n≥1} τ(n) [ (2πn)^{-s} Γ(s, 2πn) + (2πn)^{-(12-s)} Γ(12-s, 2πn) ]
# where Γ(s, x) = ∫_x^∞ t^{s-1} e^{-t} dt is the upper incomplete gamma function.
#
# This converges extremely fast since Γ(s, x) ~ x^{s-1} e^{-x} for large x.
# For n=1, x = 2π ≈ 6.28, and e^{-2π} ≈ 0.00187, so just ~100 terms should suffice.

def Lambda_Delta(s, tau_coeffs, N_terms=None):
 """Compute the completed L-function Lambda(s) for the Ramanujan Delta function.
 
 Lambda(s) = (2*pi)^{-s} * Gamma(s) * L_arith(s)
 
 Using the rapidly convergent formula:
 Lambda(s) = sum_{n=1}^{N} tau(n) * [(2*pi*n)^{-s} * Gamma_upper(s, 2*pi*n) + 
 (2*pi*n)^{-(12-s)} * Gamma_upper(12-s, 2*pi*n)]
 """
 if N_terms is None:
 N_terms = len(tau_coeffs) - 1
 
 two_pi = 2 * mpmath.pi
 s12 = 12 - s
 
 result = mpmath.mpf(0)
 for n in range(1, N_terms + 1):
 tn = tau_coeffs[n]
 if tn == 0:
 continue
 x = two_pi * n
 
 # (2*pi*n)^{-s} * Gamma_upper(s, 2*pi*n)
 term1 = mpmath.power(x, -s) * mpmath.gammainc(s, x)
 # (2*pi*n)^{-(12-s)} * Gamma_upper(12-s, 2*pi*n) 
 term2 = mpmath.power(x, -s12) * mpmath.gammainc(s12, x)
 
 result += tn * (term1 + term2)
 
 return result

# Test: Lambda(s) should be real on the real axis and at the center s=6
mpmath.mp.dps = 50
test_val = Lambda_Delta(mpmath.mpf(6), tau, N_terms=100)
print(f"Lambda(6) = {test_val}")
print(f"Im(Lambda(6)) should be ~0: {mpmath.im(test_val)}")

# Also Lambda should satisfy Lambda(s) = Lambda(12-s)
test_s = mpmath.mpf(7)
v1 = Lambda_Delta(test_s, tau, N_terms=100)
v2 = Lambda_Delta(12 - test_s, tau, N_terms=100)
print(f"\nLambda(7) = {v1}")
print(f"Lambda(5) = {v2}")
print(f"Difference = {abs(v1 - v2)}")


Lambda(6) = 0.0015448793603950272060430057803958809843299263861765
Im(Lambda(6)) should be ~0: 0.0

Lambda(7) = 0.0016339860348406993480160218298910259251323717758665
Lambda(5) = 0.0016339860348406993480160218298910259251323717758665
Difference = 0.0


In [9]:
# Perfect! Lambda(s) = Lambda(12-s) is verified exactly.
# Now let's evaluate on the critical line s = 6 + it and find zeros.
# 
# On the critical line, Lambda(6+it) should be real (for self-dual L-functions with sign +1).
# Let's verify this and then find sign changes.

# Test at the known first zero gamma_1 ≈ 9.222379
t_test = mpmath.mpf('9.222379399921102522243767192743263')
s_test = 6 + t_test * 1j
val = Lambda_Delta(s_test, tau, N_terms=200)
print(f"Lambda(6 + i*gamma_1) = {val}")
print(f"Should be ~0")
print(f"|Lambda| = {abs(val)}")
print()

# Check that Lambda is real on critical line
t_check = mpmath.mpf('5.0')
s_check = 6 + t_check * 1j
val_check = Lambda_Delta(s_check, tau, N_terms=200)
print(f"Lambda(6+5i) = {val_check}")
print(f"Real part: {mpmath.re(val_check)}")
print(f"Imag part: {mpmath.im(val_check)}")


Lambda(6 + i*gamma_1) = (1.2413108690162569357451486288855618048349541544256e-36+0j)
Should be ~0
|Lambda| = 1.2413108690162569357451486288855618048349541544256e-36

Lambda(6+5i) = (0.00035204775101650325639019064513958212927367304047239+0j)
Real part: 0.00035204775101650325639019064513958212927367304047239
Imag part: 0.0


In [10]:
# Excellent! Lambda is real on the critical line and essentially zero at gamma_1.
# Now let's systematically find all 80 zeros by detecting sign changes in Re(Lambda(6+it)).

# The function Z(t) = Re(Lambda(6+it)) is our Hardy Z-function analog.
# Zeros of Z(t) correspond to zeros of L on the critical line.

# Strategy:
# 1. Evaluate Z(t) at closely spaced points to detect sign changes
# 2. Refine each zero with bisection + Newton's method
# 3. We have 10 zeros from LMFDB to validate

# First, let's estimate where gamma_80 might be. 
# The zero counting function N(T) ~ T/(2*pi) * log(T/(2*pi*e)) for degree 2 L-functions
# For L(Delta, s), the conductor is 1 and there's a Gamma factor shift.
# Roughly, for 80 zeros we might need T ~ 150-200.

# Let's define Z(t) efficiently
def Z(t, tau_coeffs, N_terms=200):
 """Evaluate the Hardy Z-function Z(t) = Re(Lambda(6+it))."""
 s = 6 + t * 1j
 val = Lambda_Delta(s, tau_coeffs, N_terms)
 return mpmath.re(val)

# Quick scan to estimate density
print("Quick scan of Z(t) for t in [8, 40]:")
for t in range(8, 41, 1):
 t_mp = mpmath.mpf(t)
 z_val = Z(t_mp, tau, N_terms=200)
 sign = "+" if z_val > 0 else "-"
 print(f" Z({t:3d}) = {float(z_val):+.6e} [{sign}]")


Quick scan of Z(t) for t in [8, 40]:


 Z( 8) = +1.918500e-05 [+]


 Z( 9) = +1.564026e-06 [+]


 Z( 10) = -2.109865e-06 [-]


 Z( 11) = -1.548741e-06 [-]


...

 Z( 36) = -5.010733e-21 [-]


 Z( 37) = +4.354368e-22 [+]


 Z( 38) = -1.818406e-22 [-]


 Z( 39) = -2.151812e-22 [-]


 Z( 40) = -1.453309e-23 [-]


In [11]:
# Good, we can see sign changes corresponding to zeros. 
# The step size of 1 is sometimes too coarse - we might miss closely spaced zeros.
# Let's use a finer grid (step 0.2) and scan up to t ~ 200.
# 
# But first, let's think about efficiency. Each Z(t) evaluation with N_terms=200
# takes a noticeable fraction of a second. With step=0.2 and range up to ~200,
# that's ~1000 evaluations. Let's time one evaluation.

import time

mpmath.mp.dps = 40 # 40 digits should be enough for finding zeros

t0 = time.time()
for _ in range(5):
 Z(mpmath.mpf('20.0'), tau, N_terms=200)
t1 = time.time()
print(f"Average Z(t) evaluation: {(t1-t0)/5:.3f}s")

# How many terms do we actually need? For n >= some threshold, 
# Gamma_upper(s, 2*pi*n) is exponentially small.
# e^{-2*pi*n} for n=50 is already 10^{-136}. We probably only need ~30-50 terms.

t0 = time.time()
for _ in range(5):
 Z(mpmath.mpf('20.0'), tau, N_terms=50)
t1 = time.time()
print(f"Average Z(t) with N=50: {(t1-t0)/5:.3f}s")

# Check accuracy: how many terms do we need?
v200 = Z(mpmath.mpf('20.0'), tau, N_terms=200)
v50 = Z(mpmath.mpf('20.0'), tau, N_terms=50)
v30 = Z(mpmath.mpf('20.0'), tau, N_terms=30)
v20 = Z(mpmath.mpf('20.0'), tau, N_terms=20)
print(f"\nZ(20) with N=200: {v200}")
print(f"Z(20) with N=50: {v50}")
print(f"Z(20) with N=30: {v30}")
print(f"Z(20) with N=20: {v20}")
print(f"Diff 50 vs 200: {abs(v50 - v200)}")
print(f"Diff 30 vs 200: {abs(v30 - v200)}")
print(f"Diff 20 vs 200: {abs(v20 - v200)}")


Average Z(t) evaluation: 0.171s


Average Z(t) with N=50: 0.119s



Z(20) with N=200: 1.05312005770378761709867909815735700027e-11
Z(20) with N=50: 1.05312005770378761709867909815735700027e-11
Z(20) with N=30: 1.05312005770378761709867909815735700027e-11
Z(20) with N=20: 1.05312005770378761709867909815735700027e-11
Diff 50 vs 200: 0.0
Diff 30 vs 200: 0.0
Diff 20 vs 200: 0.0


In [12]:
# Amazing convergence! Even 20 terms gives identical results to 200. 
# This is because e^{-2*pi} ≈ 0.00187, so the series converges super-exponentially.
# Let's use N_terms=30 for safety and speed.

# Now for high t values, we should check if more terms are needed
# (the incomplete gamma function behavior changes with large imaginary part)
v_high_20 = Z(mpmath.mpf('150.0'), tau, N_terms=20)
v_high_50 = Z(mpmath.mpf('150.0'), tau, N_terms=50)
v_high_200 = Z(mpmath.mpf('150.0'), tau, N_terms=200)
print(f"Z(150) N=20: {v_high_20}")
print(f"Z(150) N=50: {v_high_50}")
print(f"Z(150) N=200: {v_high_200}")
print(f"Diff 20 vs 200: {abs(v_high_20 - v_high_200)}")


Z(150) N=20: -1.946671943418335154324867079404087082551e-46
Z(150) N=50: -1.946672082490544325134656318953880313579e-46
Z(150) N=200: -1.946672082490544325134656318953880313579e-46
Diff 20 vs 200: 1.390722091708097892395497932310276309361e-53


In [13]:
# Good - 50 terms is more than enough even at t=150. Let's use N_terms=50.
# 
# Now let's do a systematic scan. We need step small enough to not miss zeros.
# The average zero spacing for this L-function at height T is roughly 2*pi/log(T/(2*pi))
# At T=100, spacing ~ 2*pi/log(100/6.28) ~ 2.3. At T=200, spacing ~ 1.8.
# A step of 0.2 should be safe.

# Let's scan from t=8 to t=200 with step 0.2
# Expected time: (200-8)/0.2 * 0.12s ≈ 115 seconds

import time

mpmath.mp.dps = 40
N_TERMS = 50
STEP = mpmath.mpf('0.2')

print("Scanning for sign changes of Z(t) on [8, 200]...")
t_start_scan = time.time()

sign_changes = []
t = mpmath.mpf('8.0')
t_end = mpmath.mpf('200.0')

z_prev = Z(t, tau, N_TERMS)
t_prev = t

count = 0
t = t + STEP
while t <= t_end:
 z_curr = Z(t, tau, N_TERMS)
 
 if z_prev * z_curr < 0:
 sign_changes.append((t_prev, t, z_prev, z_curr))
 
 z_prev = z_curr
 t_prev = t
 t = t + STEP
 count += 1
 
 if count % 200 == 0:
 elapsed = time.time() - t_start_scan
 print(f" t={float(t):.1f}, found {len(sign_changes)} sign changes so far, elapsed: {elapsed:.1f}s")

elapsed = time.time() - t_start_scan
print(f"\nScan complete in {elapsed:.1f}s")
print(f"Found {len(sign_changes)} sign changes")


Scanning for sign changes of Z(t) on [8, 200]...


 t=48.2, found 18 sign changes so far, elapsed: 27.6s


 t=88.2, found 61 sign changes so far, elapsed: 67.7s


 t=128.2, found 141 sign changes so far, elapsed: 127.6s


 t=168.2, found 204 sign changes so far, elapsed: 175.5s



Scan complete in 202.9s
Found 233 sign changes


In [14]:
# We found 233 sign changes, way more than 80. Let's just refine the first 80 zeros.
# We'll use bisection first then mpmath's findroot for high-precision refinement.

print(f"First 10 sign change intervals:")
for i, (a, b, za, zb) in enumerate(sign_changes[:10]):
 print(f" {i+1}: [{float(a):.1f}, {float(b):.1f}] Z(a)={float(za):+.3e}, Z(b)={float(zb):+.3e}")


First 10 sign change intervals:
 1: [9.2, 9.4] Z(a)=+1.318e-07, Z(b)=-8.702e-07
 2: [13.8, 14.0] Z(a)=-6.676e-09, Z(b)=+4.557e-09
 3: [17.4, 17.6] Z(a)=+3.223e-11, Z(b)=-8.900e-11
 4: [19.6, 19.8] Z(a)=-2.767e-12, Z(b)=+5.619e-12
 5: [22.2, 22.4] Z(a)=+3.264e-13, Z(b)=-1.205e-13
 6: [25.2, 25.4] Z(a)=-2.581e-15, Z(b)=+3.028e-15
 7: [26.8, 27.0] Z(a)=+1.493e-17, Z(b)=-5.217e-16
 8: [28.8, 29.0] Z(a)=-1.010e-17, Z(b)=+4.189e-17
 9: [31.0, 31.2] Z(a)=+2.905e-18, Z(b)=-2.620e-19
 10: [32.6, 32.8] Z(a)=-3.004e-19, Z(b)=+3.498e-20


In [15]:
# Bisection with Z(t) evaluation is too slow. Each Z(t) call does 50 incomplete gamma
# evaluations with complex arguments at 60 dps. 200 bisection steps * 80 zeros is too many.
#
# Better approach: 
# 1. Use bisection with fewer steps just to narrow the interval
# 2. Then use mpmath.findroot with secant method for final polish
# 3. Or better: reduce the number of bisection iterations drastically
#
# Actually the main issue is 200 bisection steps is absurd. 
# For 55 digits we need about 55*3.32 = 183 bits, so ~183 bisection steps starting 
# from interval width 0.2. That's still a lot.
#
# Better strategy: use bisection for ~20 steps (gets us to ~6 digit accuracy), 
# then use secant method which converges superlinearly.

# But even 20 bisection steps * 80 zeros = 1600 Z evaluations * ~0.2s = 320s
# Plus secant iterations.

# Let me try a more efficient approach: use mpmath.findroot directly with 
# good starting points from the sign change intervals.

mpmath.mp.dps = 50

def Z_simple(t):
 """Z(t) for use with findroot."""
 s = 6 + t * 1j
 val = Lambda_Delta(s, tau, N_terms=50)
 return mpmath.re(val)

# Try using mpmath.findroot with the Illinois method (modified regula falsi)
# which is much faster than pure bisection
import time

# First let's time a single findroot call
t0 = time.time()
# Use the midpoint as starting guess
a, b = sign_changes[0][0], sign_changes[0][1]
mid = (a + b) / 2
zero_test = mpmath.findroot(Z_simple, (float(a), float(b)), solver='illinois', tol=mpmath.mpf(10)**(-45))
t1 = time.time()
print(f"findroot took {t1-t0:.2f}s")
print(f"Zero found: {mpmath.nstr(zero_test, 40)}")
print(f"Known: 9.222379399921102522243767192743263")
print(f"Z(zero) = {Z_simple(zero_test)}")


findroot took 2.73s
Zero found: 9.222379399921102522243767192743478135529
Known: 9.222379399921102522243767192743263
Z(zero) = -8.5318837862599980950560899271056814436496143654941e-52


In [16]:
# The findroot gives us the zero, but precision diverges from LMFDB at digit ~31.
# This is because LMFDB shows 33 digits and our computation has 50 dps.
# The agreement is excellent through 30 digits:
# Ours: 9.222379399921102522243767192743478
# LMFDB: 9.222379399921102522243767192743263
# Differs at position 31. Let's increase precision.

mpmath.mp.dps = 60

def Z_hp(t):
 """Z(t) at high precision."""
 s = 6 + t * 1j
 val = Lambda_Delta(s, tau, N_terms=50)
 return mpmath.re(val)

t0 = time.time()
a, b = sign_changes[0][0], sign_changes[0][1]
zero_test = mpmath.findroot(Z_hp, (float(a), float(b)), solver='illinois', tol=mpmath.mpf(10)**(-55))
t1 = time.time()
print(f"findroot at dps=60 took {t1-t0:.2f}s")
print(f"Zero: {mpmath.nstr(zero_test, 45)}")
print(f"LMFDB: 9.222379399921102522243767192743263")


findroot at dps=60 took 4.36s
Zero: 9.22237939992110252224376719274347813552877062
LMFDB: 9.222379399921102522243767192743263


In [17]:
# Hmm, we get 9.222379399921102522243767192743478... while LMFDB says 9.222379399921102522243767192743263.
# Disagreement at digit 31. Let me check our computation more carefully.
# Maybe we need even more tau coefficients for higher accuracy? Or higher dps?

mpmath.mp.dps = 80

# Check convergence of the zero with different numbers of terms
for N in [30, 50, 100, 200, 400]:
 def Z_test(t, N_terms=N):
 s = 6 + t * 1j
 val = Lambda_Delta(s, tau, N_terms=N_terms)
 return mpmath.re(val)
 
 a, b = mpmath.mpf('9.2'), mpmath.mpf('9.4')
 try:
 zero = mpmath.findroot(Z_test, (9.2, 9.4), solver='illinois', tol=mpmath.mpf(10)**(-70))
 print(f"N={N:3d}: {mpmath.nstr(zero, 50)}")
 except Exception as e:
 print(f"N={N:3d}: Error: {e}")


N= 30: 9.222379399921102522243767192743478135528770622432


N= 50: 9.222379399921102522243767192743478135528770622432


N=100: 9.222379399921102522243767192743478135528770622432


N=200: 9.222379399921102522243767192743478135528770622432


N=400: 9.222379399921102522243767192743478135528770622432


In [18]:
# Our computation is fully converged: 9.22237939992110252224376719274347813552877...
# The LMFDB value is: 9.222379399921102522243767192743263
# 
# They agree through 30 digits, then differ. Let me check: could the LMFDB value 
# be truncated rather than rounded? Or could there be a normalization difference?
# 
# Let me cross-check with another source. Let me search for known zeros of the 
# Ramanujan L-function.

# Actually, let me check: maybe LMFDB stores only ~30 digits of precision and 
# the trailing digits are inaccurate. Our value with 80 dps is:
# 9.222379399921102522243767192743478135528770622432
# vs LMFDB's 33 digits:
# 9.222379399921102522243767192743263
#
# First 30 digits match: 9.22237939992110252224376719274
# Then ours: 3478... vs LMFDB: 3263...
# That's a real discrepancy at digit 31.
#
# Let me try to verify with a completely independent computation using more tau coefficients.
# Actually, wait - let me recompute tau coefficients to make sure they're correct.

# Verify more tau values against known values
known_tau = {
 1: 1, 2: -24, 3: 252, 4: -1472, 5: 4830,
 6: -6048, 7: -16744, 8: 84480, 9: -113643, 10: -115920,
 11: 534612, 12: -370944, 13: -577738, 14: 401856, 15: 1217160,
 16: 987136, 17: -6905934, 18: 2727432, 19: 10661420, 20: -7109760
}

all_match = True
for n, expected in known_tau.items():
 if tau[n] != expected:
 print(f"MISMATCH at tau({n}): got {tau[n]}, expected {expected}")
 all_match = False

if all_match:
 print("All tau coefficients match known values ✓")

# Also check tau(23) which should be 18643272
print(f"tau(23) = {tau[23]}")
print(f"Expected: 18643272")


All tau coefficients match known values ✓
tau(23) = 18643272
Expected: 18643272


In [19]:
# tau coefficients are correct. Let me also verify by computing L_arith(12)
# which should have a known value related to the Ramanujan conjecture.
# Actually, let me just check against a known L-function value.

# L_arith(12) = sum tau(n)/n^12 
# Actually for s > 13/2 = 6.5, the Dirichlet series converges.
# L_arith(12) should converge well.

mpmath.mp.dps = 50
L_12 = sum(mpmath.mpf(tau[n]) / mpmath.power(n, 12) for n in range(1, 801))
print(f"L_arith(12) ≈ {L_12}")

# Known: L(Delta, 12) = (2^10 * 3^5 * 5^2 * 7 * 691) / (2^{15} * 3^6 * 5^3 * 7 * 691) * ...
# Actually, the value at integer points involves Bernoulli numbers.
# L(Delta, 12) * (2*pi)^{-12} * Gamma(12) = Lambda(12)

Lambda_12 = Lambda_Delta(mpmath.mpf(12), tau, N_terms=50)
print(f"Lambda(12) = {Lambda_12}")

# From functional equation, Lambda(12) = Lambda(0). Let's check.
Lambda_0 = Lambda_Delta(mpmath.mpf('0.01'), tau, N_terms=50) # near 0
print(f"Lambda(0.01) = {Lambda_0}")

# OK the issue might just be that LMFDB rounds or truncates at 30-ish digits.
# Our computation is stable and converged. Let me trust our computation.
# 
# Actually wait - let me look at this from another angle. Let me check if
# LMFDB might be using a different normalization or definition.
# Their "analytic_normalization" field says 5.5, confirming s -> s + 5.5 = s + 11/2.
# So in their analytic normalization, zeros are at 1/2 + i*gamma, and
# gamma values match our computation of zeros of Lambda(6+it).
# The agreement through 30 digits confirms this is correct.
# The small discrepancy at digit 31+ is likely LMFDB's precision limitation.
print("\nConclusion: Our zero value is more precise than LMFDB's stored value.")
print("Agreement through 30 digits confirms correctness.")


L_arith(12) ≈ 0.99454369291844374878512053422914446540639312948464
Lambda(12) = 0.010486273129241151969641520292195117893626925889443
Lambda(0.01) = 0.010422708156603505568039753392712069333417624689241

Conclusion: Our zero value is more precise than LMFDB's stored value.
Agreement through 30 digits confirms correctness.


In [20]:
# Now let's compute all 80 zeros efficiently. Each findroot call takes ~4-5s at dps=60.
# 80 zeros * 5s = 400s ≈ 7 minutes. That should be fine within the timeout.
# But let me try to parallelize or speed up.

# Actually, let me use a two-phase approach:
# Phase 1: Quick refinement at low dps to get ~15 digit accuracy (fast)
# Phase 2: Final refinement at high dps to get 35+ digit accuracy

# But the illinois solver in mpmath should handle this reasonably.
# Let me try with dps=50 (which gives ~45 correct digits) and see timing.

mpmath.mp.dps = 50

import time

# First, let's quickly verify: how long does findroot take for various zeros?
for i in [0, 20, 40, 60, 79]:
 if i >= len(sign_changes):
 break
 a, b = sign_changes[i][0], sign_changes[i][1]
 t0 = time.time()
 zero = mpmath.findroot(Z_hp, (float(a), float(b)), solver='illinois', tol=mpmath.mpf(10)**(-45))
 t1 = time.time()
 print(f"Zero {i+1}: {t1-t0:.2f}s gamma = {mpmath.nstr(zero, 40)}")


Zero 1: 2.79s gamma = 9.222379399921102522243767192743478135529


Zero 21: 3.45s gamma = 51.15656028143634522883873402450688406786


Zero 41: 1.81s gamma = 77.12732773293719382361911333967191257493


Zero 61: 1.88s gamma = 86.72806145647480022505789890245540119621


Zero 80: 2.07s gamma = 97.29379684797956263074346095550385574139


In [21]:
# Each zero takes ~2-3.5s. 80 zeros ~ 200-250s. Let's do it.
# Actually let me try a strategy: use the bisection approach manually with fewer steps
# to narrow down, then one final secant step.
# 
# Or just use illinois solver directly which is already fast enough.

mpmath.mp.dps = 50

def Z_for_root(t):
 s = 6 + t * 1j
 val = Lambda_Delta(s, tau, N_terms=50)
 return mpmath.re(val)

print("Computing 80 zeros of L(Delta, s) on the critical line...")
print("(analytic normalization: zeros at 1/2 + i*gamma)")
print()

zeros_list = []
t_total_start = time.time()

for i in range(80):
 a_val, b_val = float(sign_changes[i][0]), float(sign_changes[i][1])
 
 zero = mpmath.findroot(Z_for_root, (a_val, b_val), solver='illinois', tol=mpmath.mpf(10)**(-45))
 zeros_list.append(zero)
 
 if (i + 1) % 10 == 0:
 elapsed = time.time() - t_total_start
 eta = elapsed / (i + 1) * 80
 print(f" {i+1}/80 done, elapsed: {elapsed:.1f}s, ETA total: {eta:.0f}s")
 print(f" γ_{i+1} = {mpmath.nstr(zero, 40)}")

total = time.time() - t_total_start
print(f"\nAll 80 zeros computed in {total:.1f}s")


Computing 80 zeros of L(Delta, s) on the critical line...
(analytic normalization: zeros at 1/2 + i*gamma)



 10/80 done, elapsed: 33.0s, ETA total: 264s
 γ_10 = 32.7748753822312074418304556733119899991


 20/80 done, elapsed: 68.1s, ETA total: 272s
 γ_20 = 49.27605353655817840918763046889689170684


 30/80 done, elapsed: 102.1s, ETA total: 272s
 γ_30 = 64.08664571913868223421070317325958296997


 40/80 done, elapsed: 124.7s, ETA total: 249s
 γ_40 = 77.06013772082039848977178300548120157023


 50/80 done, elapsed: 142.9s, ETA total: 229s
 γ_50 = 81.91069903016953843464831472240731244847


 60/80 done, elapsed: 161.5s, ETA total: 215s
 γ_60 = 86.70921256150492041890117595345676327136


 70/80 done, elapsed: 181.1s, ETA total: 207s
 γ_70 = 92.39381047936055854248303374319811667793


 80/80 done, elapsed: 201.0s, ETA total: 201s
 γ_80 = 97.29379684797956263074346095550385574139

All 80 zeros computed in 201.0s


In [22]:
# Great! All 80 zeros computed. Now let's verify against LMFDB values.

print("Verification against LMFDB values:")
print("=" * 80)
lmfdb_zeros_str = [
 '9.222379399921102522243767192743263',
 '13.90754986139213440644668132877006',
 '17.44277697823447331355152513712760',
 '19.65651314195496100012728175632157',
 '22.33610363720986727568267445923646',
 '25.27463654811236535674532419313339',
 '26.80439115835040303257574923358447',
 '28.83168262418687544502196191298445',
 '31.17820949836025906449218889077411',
 '32.77487538223120744183045567331179',
]

for i, lmfdb_str in enumerate(lmfdb_zeros_str):
 lmfdb_val = mpmath.mpf(lmfdb_str)
 our_val = zeros_list[i]
 diff = abs(our_val - lmfdb_val)
 our_str = mpmath.nstr(our_val, 40)
 print(f"γ_{i+1:2d}:")
 print(f" Ours: {our_str}")
 print(f" LMFDB: {lmfdb_str}")
 print(f" |diff| = {mpmath.nstr(diff, 5)}")
 # Count matching digits
 matching = 0
 for j in range(min(len(lmfdb_str), len(our_str))):
 if j < len(lmfdb_str) and j < len(our_str) and lmfdb_str[j] == our_str[j]:
 matching += 1
 else:
 break
 print(f" Matching leading chars: {matching}")
 print()


Verification against LMFDB values:
γ_ 1:
 Ours: 9.222379399921102522243767192743478135529
 LMFDB: 9.222379399921102522243767192743263
 |diff| = 2.1514e-31
 Matching leading chars: 32

γ_ 2:
 Ours: 13.90754986139213440644668132877021949176
 LMFDB: 13.90754986139213440644668132877006
 |diff| = 1.5949e-31
 Matching leading chars: 33

γ_ 3:
 Ours: 17.44277697823447331355152513712726271977
 LMFDB: 17.44277697823447331355152513712760
 |diff| = 3.3728e-31
 Matching leading chars: 33

γ_ 4:
 Ours: 19.65651314195496100012728175632130280163
 LMFDB: 19.65651314195496100012728175632157
 |diff| = 2.672e-31
 Matching leading chars: 33

γ_ 5:
 Ours: 22.33610363720986727568267445923624619252
 LMFDB: 22.33610363720986727568267445923646
 |diff| = 2.1381e-31
 Matching leading chars: 33

γ_ 6:
 Ours: 25.2746365481123653567453241931334631186
 LMFDB: 25.27463654811236535674532419313339
 |diff| = 7.3119e-32
 Matching leading chars: 33

γ_ 7:
 Ours: 26.80439115835040303257574923358456474715
 LMFDB: 26.8043911

In [23]:
# Agreement through ~30-31 digits with LMFDB, with differences of order 10^{-31} to 10^{-32}.
# This is consistent: LMFDB stores ~33 digits but the last 2-3 digits may have rounding.
# Our computation at dps=50 should have ~45 correct digits.
# 
# The differences are at the 31st-32nd digit which is near LMFDB's precision limit.
# Both computations agree to 30+ digits which is excellent validation.
#
# However, to be extra safe, let me increase our precision to dps=70 for the final output
# to ensure we have 35+ correct digits. But first let me check: at dps=50, how many 
# digits are actually correct?

# Let me recompute gamma_1 at higher precision and compare
mpmath.mp.dps = 70

def Z_hp70(t):
 s = 6 + t * 1j
 val = Lambda_Delta(s, tau, N_terms=50)
 return mpmath.re(val)

zero_1_hp = mpmath.findroot(Z_hp70, (9.2, 9.4), solver='illinois', tol=mpmath.mpf(10)**(-65))
print(f"γ_1 at dps=70: {mpmath.nstr(zero_1_hp, 60)}")
print(f"γ_1 at dps=50: {mpmath.nstr(zeros_list[0], 50)}")
# Check agreement
diff = abs(zero_1_hp - zeros_list[0])
print(f"Difference: {mpmath.nstr(diff, 5)}")


γ_1 at dps=70: 9.22237939992110252224376719274347813552877062243200928999819
γ_1 at dps=50: 9.2223793999211025222437671927434781355287706225799
Difference: 1.4789e-46


In [24]:
# Our dps=50 computation agrees with dps=70 through 46 digits. So at dps=50 we 
# have ~46 correct digits. Since we need 30+, this is more than sufficient.
# The LMFDB values have ~30 correct digits.
#
# Let me now save the results. We need:
# 1. Text file with one float per line, 30+ digit precision
# 2. Pickle file with mpmath.mpf objects

outdir = "/workspace/98359772-f251-4284-9fcb-d126816adbeb"

# Save text file
txt_path = f"{outdir}/LDelta_zeros_80.txt"
with open(txt_path, 'w') as f:
 for z in zeros_list:
 # Output 40 significant digits (we have ~46 correct digits at dps=50)
 f.write(mpmath.nstr(z, 40, strip_zeros=False) + '\n')

print(f"Saved {txt_path}")

# Verify
with open(txt_path) as f:
 lines = f.readlines()
print(f"Number of lines: {len(lines)}")
print(f"First line: {lines[0].strip()}")
print(f"Last line: {lines[-1].strip()}")

# Check digit count
first_val = lines[0].strip()
print(f"\nDigits in first value: {len(first_val.replace('.', ''))}")


Saved /workspace/98359772-f251-4284-9fcb-d126816adbeb/LDelta_zeros_80.txt
Number of lines: 80
First line: 9.222379399921102522243767192743478135529
Last line: 97.29379684797956263074346095550385574139

Digits in first value: 40


In [25]:
# Save pickle file
import pickle

pkl_path = f"{outdir}/LDelta_zeros_80.pkl"
with open(pkl_path, 'wb') as f:
 pickle.dump(zeros_list, f)

print(f"Saved {pkl_path}")

# Verify pickle
with open(pkl_path, 'rb') as f:
 loaded = pickle.load(f)

print(f"Loaded {len(loaded)} mpf values from pickle")
print(f"Type: {type(loaded[0])}")
print(f"First: {loaded[0]}")
print(f"Last: {loaded[-1]}")
print(f"Match: {loaded[0] == zeros_list[0] and loaded[-1] == zeros_list[-1]}")


Saved /workspace/98359772-f251-4284-9fcb-d126816adbeb/LDelta_zeros_80.pkl
Loaded 80 mpf values from pickle
Type: <class 'mpmath.ctx_mp_python.mpf'>
First: 9.222379399921102522243767192743478135528770622579896205250912687267636
Last: 97.29379684797956263074346095550385574138888193184421473890853889432447
Match: True


In [26]:
# Now let's print the final report

print("=" * 80)
print("ZEROS OF L(Δ, s) — RAMANUJAN TAU L-FUNCTION")
print("Analytic normalization: zeros at ρ = 1/2 + iγ")
print("Weight 12, Level 1")
print("=" * 80)

print("\n--- First 5 zeros ---")
for i in range(5):
 print(f" γ_{i+1:2d} = {mpmath.nstr(zeros_list[i], 40)}")

print("\n--- Last 5 zeros ---")
for i in range(75, 80):
 print(f" γ_{i+1:2d} = {mpmath.nstr(zeros_list[i], 40)}")

print(f"\n--- γ_80 ---")
print(f" γ_80 = {mpmath.nstr(zeros_list[79], 45)}")

print("\n--- Verification ---")
print(f" γ_1 = {mpmath.nstr(zeros_list[0], 40)}")
print(f" Expected: ≈ 9.22237... ✓ (matches through 30+ digits with LMFDB)")
print(f" γ_2 = {mpmath.nstr(zeros_list[1], 40)}")
print(f" Expected: ≈ 13.907... ✓ (matches through 30+ digits with LMFDB)")

print(f"\n--- Statistics ---")
print(f" Total zeros computed: {len(zeros_list)}")
print(f" Precision: ~40 significant digits per zero")
print(f" Computation method: Completed L-function Λ(s) = Σ τ(n)[(2πn)^{{-s}}Γ(s,2πn) + (2πn)^{{-(12-s)}}Γ(12-s,2πn)]")
print(f" Tau coefficients: {len(tau)-1} (computed from q-expansion of Δ)")
print(f" Series terms used: 50 (convergence verified)")
print(f" mpmath dps: 50 (verified ~46 correct digits)")
print(f" Zero finder: Illinois method (modified regula falsi)")
print(f" Sign change detection: grid scan with step 0.2 on [8, 200]")

print(f"\n--- Output files ---")
print(f" {outdir}/LDelta_zeros_80.txt (text, one float per line)")
print(f" {outdir}/LDelta_zeros_80.pkl (pickle, list of mpmath.mpf)")


ZEROS OF L(Δ, s) — RAMANUJAN TAU L-FUNCTION
Analytic normalization: zeros at ρ = 1/2 + iγ
Weight 12, Level 1

--- First 5 zeros ---
 γ_ 1 = 9.222379399921102522243767192743478135529
 γ_ 2 = 13.90754986139213440644668132877021949176
 γ_ 3 = 17.44277697823447331355152513712726271977
 γ_ 4 = 19.65651314195496100012728175632130280163
 γ_ 5 = 22.33610363720986727568267445923624619252

--- Last 5 zeros ---
 γ_76 = 94.94352854880191456956735140848434832143
 γ_77 = 99.55715035208027011610192322009630277937
 γ_78 = 97.19100640910648129641378831762031311114
 γ_79 = 97.21828844390408052736776635859734571891
 γ_80 = 97.29379684797956263074346095550385574139

--- γ_80 ---
 γ_80 = 97.2937968479795626307434609555038557413888819

--- Verification ---
 γ_1 = 9.222379399921102522243767192743478135529
 Expected: ≈ 9.22237... ✓ (matches through 30+ digits with LMFDB)
 γ_2 = 13.90754986139213440644668132877021949176
 Expected: ≈ 13.907... ✓ (matches through 30+ digits with LMFDB)

--- Statistics ---
 Total

In [27]:
# Wait - I notice the last 5 zeros aren't in sorted order! γ_76 > γ_77 and γ_78 < γ_76.
# This is because the sign_changes list is ordered by the t-values where sign changes 
# were detected, and nearby zeros can appear in slightly different order.
# Let me sort all zeros and re-save.

zeros_sorted = sorted(zeros_list)

# Check if they were already sorted
was_sorted = all(zeros_sorted[i] <= zeros_sorted[i+1] for i in range(len(zeros_sorted)-1))
original_sorted = all(zeros_list[i] <= zeros_list[i+1] for i in range(len(zeros_list)-1))

print(f"Original list sorted? {original_sorted}")
print(f"After sorting: {was_sorted}")

# Find where the order was wrong
for i in range(len(zeros_list)-1):
 if zeros_list[i] > zeros_list[i+1]:
 print(f" Out of order at index {i}: γ_{i+1}={float(zeros_list[i]):.4f} > γ_{i+2}={float(zeros_list[i+1]):.4f}")

print("\nSorted last 10 zeros:")
for i in range(70, 80):
 print(f" γ_{i+1:2d} = {mpmath.nstr(zeros_sorted[i], 40)}")


Original list sorted? False
After sorting: True
 Out of order at index 44: γ_45=82.0226 > γ_46=80.5648
 Out of order at index 45: γ_46=80.5648 > γ_47=80.5446
 Out of order at index 46: γ_47=80.5446 > γ_48=80.0442
 Out of order at index 48: γ_49=83.0475 > γ_50=81.9107
 Out of order at index 49: γ_50=81.9107 > γ_51=81.8631
 Out of order at index 50: γ_51=81.8631 > γ_52=79.2278
 Out of order at index 53: γ_54=84.0611 > γ_55=83.9574
 Out of order at index 56: γ_57=91.1446 > γ_58=85.4772
 Out of order at index 57: γ_58=85.4772 > γ_59=85.2812
 Out of order at index 61: γ_62=88.0953 > γ_63=87.8219
 Out of order at index 63: γ_64=93.3608 > γ_65=90.3711
 Out of order at index 67: γ_68=93.3859 > γ_69=92.4460
 Out of order at index 68: γ_69=92.4460 > γ_70=92.3938
 Out of order at index 72: γ_73=95.3201 > γ_74=94.9573
 Out of order at index 74: γ_75=94.9864 > γ_76=94.9435
 Out of order at index 76: γ_77=99.5572 > γ_78=97.1910

Sorted last 10 zeros:
 γ_71 = 93.36082778892856327041977493741365600965

In [28]:
# Good, now let's re-save with sorted zeros.
zeros_list = zeros_sorted

# Save text file
txt_path = f"{outdir}/LDelta_zeros_80.txt"
with open(txt_path, 'w') as f:
 for z in zeros_list:
 f.write(mpmath.nstr(z, 40, strip_zeros=False) + '\n')

print(f"Saved sorted zeros to {txt_path}")

# Save pickle file 
pkl_path = f"{outdir}/LDelta_zeros_80.pkl"
with open(pkl_path, 'wb') as f:
 pickle.dump(zeros_list, f)

print(f"Saved sorted zeros to {pkl_path}")

# Verify
with open(txt_path) as f:
 lines = f.readlines()
print(f"\nLines in text file: {len(lines)}")

# Final report
print("\n" + "=" * 80)
print("FINAL SORTED ZEROS OF L(Δ, s)")
print("=" * 80)

print("\n--- First 5 zeros ---")
for i in range(5):
 print(f" γ_{i+1:2d} = {mpmath.nstr(zeros_list[i], 40)}")

print("\n--- Last 5 zeros ---")
for i in range(75, 80):
 print(f" γ_{i+1:2d} = {mpmath.nstr(zeros_list[i], 40)}")

print(f"\n γ_80 = {mpmath.nstr(zeros_list[79], 45)}")

# Verify γ_1 and γ_2
print(f"\n--- Verification ---")
print(f" γ_1 = {mpmath.nstr(zeros_list[0], 35)}")
print(f" Expected ≈ 9.22237... ✓")
print(f" γ_2 = {mpmath.nstr(zeros_list[1], 35)}")
print(f" Expected ≈ 13.907... ✓")


Saved sorted zeros to /workspace/98359772-f251-4284-9fcb-d126816adbeb/LDelta_zeros_80.txt
Saved sorted zeros to /workspace/98359772-f251-4284-9fcb-d126816adbeb/LDelta_zeros_80.pkl

Lines in text file: 80

FINAL SORTED ZEROS OF L(Δ, s)

--- First 5 zeros ---
 γ_ 1 = 9.222379399921102522243767192743478135529
 γ_ 2 = 13.90754986139213440644668132877021949176
 γ_ 3 = 17.44277697823447331355152513712726271977
 γ_ 4 = 19.65651314195496100012728175632130280163
 γ_ 5 = 22.33610363720986727568267445923624619252

--- Last 5 zeros ---
 γ_76 = 95.3201040690215024949567564995845256037
 γ_77 = 97.19100640910648129641378831762031311114
 γ_78 = 97.21828844390408052736776635859734571891
 γ_79 = 97.29379684797956263074346095550385574139
 γ_80 = 99.55715035208027011610192322009630277937

 γ_80 = 99.5571503520802701161019232200963027793652979

--- Verification ---
 γ_1 = 9.2223793999211025222437671927434781
 Expected ≈ 9.22237... ✓
 γ_2 = 13.907549861392134406446681328770219
 Expected ≈ 13.907... ✓


In [29]:
# Let me also verify the complete file contents
with open(txt_path) as f:
 lines = f.readlines()

print("Complete list of 80 zeros (sorted):")
print("-" * 50)
for i, line in enumerate(lines):
 print(f"γ_{i+1:2d} = {line.strip()}")


Complete list of 80 zeros (sorted):
--------------------------------------------------
γ_ 1 = 9.222379399921102522243767192743478135529
γ_ 2 = 13.90754986139213440644668132877021949176
γ_ 3 = 17.44277697823447331355152513712726271977
γ_ 4 = 19.65651314195496100012728175632130280163
γ_ 5 = 22.33610363720986727568267445923624619252
γ_ 6 = 25.27463654811236535674532419313346311860
γ_ 7 = 26.80439115835040303257574923358456474715
γ_ 8 = 28.83168262418687544502196191298438972569
γ_ 9 = 31.17820949836025906449218889077405585465
γ_10 = 32.77487538223120744183045567331198999910
γ_11 = 35.19699584121007259421991908434915014457
γ_12 = 36.74146297671030649582326496066147677632
γ_13 = 37.75391597562427047841879244534662186252
γ_14 = 40.21903437422132048462923212843893000885
γ_15 = 41.73049228930784883112244640313318958457
γ_16 = 43.59174123557517033366400266031092495435
γ_17 = 45.04007921377559647602260039696535950183
γ_18 = 46.19731875314330959537038228741598531328
γ_19 = 48.359052478023669542644

In [30]:
# Let me check: are there really 80 zeros up to ~99.56? Let me verify using the 
# zero counting formula. The number of zeros with 0 < gamma < T for this L-function is
# approximately N(T) = T/(2*pi) * log(T/(2*pi*e)) + O(log T) for a degree 2 L-function
# with conductor 1.
#
# More precisely, for L(Delta, s) with Gamma_R(s+11/2) factor:
# N(T) ≈ T/pi * log(T/(2*pi)) - T/pi + 7/8 + S(T)/pi
# where S(T) is the argument of the L-function on the critical line.
# 
# For our case, with Gamma(s+11/2) = Gamma_C(s+11/2) ~ Gamma_R(s+11) Gamma_R(s+12):
# Actually let me just check with the LMFDB's plot_delta and plot_values...

# Actually, the scan from 8 to 100 found zeros. Let me count how many sign changes
# we found up to ~100:
count_to_100 = sum(1 for a, b, _, _ in sign_changes if float(a) < 100)
print(f"Sign changes with a < 100: {count_to_100}")

# And we need 80 zeros. Let me also verify we haven't missed any by checking
# that no interval is suspiciously large between consecutive sorted zeros.
print("\nLargest gaps between consecutive zeros:")
gaps = [(float(zeros_list[i+1] - zeros_list[i]), i+1, i+2) for i in range(79)]
gaps.sort(reverse=True)
for gap, i, j in gaps[:10]:
 print(f" Gap {gap:.4f} between γ_{i} and γ_{j}")

# The average gap should decrease with height. For zeros near T ~ 80-100,
# the expected average gap is about 2*pi / log(T/(2*pi*e)) ≈ 2-2.5
# We see some larger gaps which could indicate missed zeros, but also 
# many very close pairs.


Sign changes with a < 100: 85

Largest gaps between consecutive zeros:
 Gap 4.6852 between γ_1 and γ_2
 Gap 3.5352 between γ_2 and γ_3
 Gap 2.9385 between γ_5 and γ_6
 Gap 2.6796 between γ_4 and γ_5
 Gap 2.4651 between γ_13 and γ_14
 Gap 2.4221 between γ_10 and γ_11
 Gap 2.3465 between γ_8 and γ_9
 Gap 2.2758 between γ_62 and γ_63
 Gap 2.2634 between γ_79 and γ_80
 Gap 2.2137 between γ_3 and γ_4


In [31]:
# We found 85 sign changes up to t=100 but only took the first 80 from the unsorted list.
# After sorting, all 80 are the smallest 80 zeros. Let me verify this by checking 
# that the 5 sign changes we skipped (indices 80-84) all have t > γ_80 ≈ 99.56.
# Actually wait - we found 233 sign changes total up to t=200, and we took the first 80
# from that list (ordered by detection position, not by zero value). So some of the 
# "first 80 detected" might actually be larger than some later-detected ones.
# 
# Let me re-check: did we get the CORRECT 80 smallest zeros? We should refine ALL 
# sign changes up to some t_max and then sort and take the smallest 80.
# 
# Actually no - sign changes are detected in order of t, and the intervals don't overlap
# much. The only issue is that our grid step of 0.2 might place a sign change at a slightly
# different position. But the zero values are correct and the sorted list gives the right ordering.
#
# Let me verify: count distinct zeros below 100 among our 80 sorted zeros.
count_below_100 = sum(1 for z in zeros_list if float(z) < 100)
print(f"Zeros below 100 in our list: {count_below_100}")

# And count sign changes we found below 100:
sc_below_100 = sum(1 for a, b, _, _ in sign_changes if float(b) < 100)
print(f"Sign changes with b < 100: {sc_below_100}")

# If we found 85 sign changes below t=100, but only kept 80, 
# did we miss 5 zeros from the 80 smallest?
# Actually the issue is simpler: we detected sign changes in order, and the
# grid might sometimes detect a sign change in interval [a, a+0.2] where the 
# actual zero is at the upper end near a+0.2, and the next sign change might 
# be for a zero that's actually smaller. But this shouldn't happen because 
# both the grid evaluation and the zeros increase monotonically... 
# unless there are very closely spaced zeros where the function oscillates 
# within a single grid step.
# 
# Looking at the data, many zeros above t=79 are very close together (gaps < 0.1).
# Our step of 0.2 might miss some. But we DID detect them (85 changes below 100).
# The issue is we only refined the first 80 in detection order, which is indeed 
# the first 80 zeros encountered in the scan.
#
# Wait - actually ALL sign changes are detected in ascending t order, so the first
# 80 detected are the 80 zeros with smallest t values. After sorting the refined
# values, the order might shift slightly due to the refinement, but all 80 should 
# be below any zero not in the list.
#
# Actually, upon reflection, the detection and sorting is correct because:
# 1. Sign changes in [a_i, b_i] are found in order i=1,2,...
# 2. a_1 < a_2 < ... (grid increases monotonically) 
# 3. Each zero is in [a_i, b_i] = [a_i, a_i + 0.2]
# 4. So zero_i ∈ [a_i, a_i+0.2] and zero_{i+1} ∈ [a_{i+1}, a_{i+1}+0.2]
# 5. If a_i < a_{i+1}, then zero_i < a_{i+1} + 0.2 = b_{i+1}
# 6. But zero_i could be > zero_{i+1} if zero_i ≈ a_i + 0.2 and zero_{i+1} ≈ a_{i+1}
# and a_{i+1} = a_i + 0.2 (consecutive grid points with no gap)
# 
# This is exactly what happened: closely spaced zeros cause the ordering to 
# be slightly off, but after sorting we have the correct 80 smallest zeros.
# All 80 refined zeros are below max(b_80) = sign_changes[79][1].
print(f"Upper bound of 80th interval: {float(sign_changes[79][1]):.2f}")
# The 81st detection: 
print(f"Lower bound of 81st interval: {float(sign_changes[80][0]):.2f}")
# So the 81st zero is > sign_changes[80][0]
# And all our 80 zeros are < sign_changes[79][1]
# Since sign_changes[80][0] >= sign_changes[79][1] - 0.2, this is fine.

# The question is whether there could be a zero in [sign_changes[79][1], sign_changes[80][0]]
# that we missed. Let me check.
gap_80_81 = float(sign_changes[80][0]) - float(sign_changes[79][1])
print(f"Gap between 80th and 81st sign change intervals: {gap_80_81:.2f}")

# If there's no sign change in this gap, we're fine.
# Let's verify: evaluate Z at a few points in this gap.
a80 = sign_changes[79][1]
a81 = sign_changes[80][0]
print(f"\nChecking for missing zeros between {float(a80):.2f} and {float(a81):.2f}:")
t_check = a80
while t_check <= a81:
 z_val = Z_for_root(t_check)
 print(f" Z({float(t_check):.2f}) = {float(z_val):+.3e}")
 t_check += mpmath.mpf('0.1')


Zeros below 100 in our list: 80
Sign changes with b < 100: 85
Upper bound of 80th interval: 97.60
Lower bound of 81st interval: 97.60
Gap between 80th and 81st sign change intervals: 0.00

Checking for missing zeros between 97.60 and 97.60:


 Z(97.60) = -1.796e-60


In [32]:
# The 80th and 81st sign change intervals are contiguous (no gap). 
# All 80 zeros below 100 are in our list.
# 
# But wait - we found 85 sign changes below t=100, and we only took the first 80 
# detection-order sign changes. After refining those 80 and sorting, all 80 are below 100.
# The remaining 5 sign changes (indices 80-84) also have zeros below 100, but those 
# zeros are LARGER than our 80th zero.
# 
# Let me verify: the 81st sign change interval is at:
print(f"81st sign change: [{float(sign_changes[80][0]):.2f}, {float(sign_changes[80][1]):.2f}]")
# This zero would be the 81st zero ≈ between those bounds.
# Our γ_80 (sorted) = 99.557, and the 81st zero is somewhere around ~97.8.
# Wait, that can't be right...

# Let me check more carefully
for i in range(78, 86):
 if i < len(sign_changes):
 a, b = sign_changes[i][0], sign_changes[i][1]
 print(f"Sign change {i+1}: [{float(a):.2f}, {float(b):.2f}]")

# If sign change 81 is at [97.60, 97.80], and our gamma_80 (sorted) = 99.56,
# then we're missing a zero at ~97.7 that should be between gamma_79 and gamma_80!


81st sign change: [97.60, 97.80]
Sign change 79: [96.80, 97.00]
Sign change 80: [97.40, 97.60]
Sign change 81: [97.60, 97.80]
Sign change 82: [97.80, 98.00]
Sign change 83: [98.00, 98.20]
Sign change 84: [98.60, 98.80]
Sign change 85: [99.00, 99.20]
Sign change 86: [100.40, 100.60]


In [33]:
# Hmm! We have sign changes at indices 79-85 that are all below t=100:
# 79: [96.80, 97.00] -> this is our 80th detection, gamma ≈ 97.19 or so
# 80: [97.40, 97.60] -> this should give us another zero around ~97.5
# 81: [97.60, 97.80] -> another zero ~97.7
# 82: [97.80, 98.00] -> another zero ~97.9
# 83: [98.00, 98.20] -> another zero ~98.1
# 84: [98.60, 98.80] -> another zero ~98.7
# 85: [99.00, 99.20] -> this gives gamma ≈ 99.1 or similar

# Wait, but we already included sign_changes[0:80], which are the first 80 sign changes.
# Let me check what the last few of those first 80 are:
for i in range(75, 80):
 a, b = sign_changes[i][0], sign_changes[i][1]
 print(f"Sign change {i+1}: [{float(a):.2f}, {float(b):.2f}]")

# Now the SORTED zeros list shows gamma_80 = 99.557. But if sign change 80 
# (the 80th detected) is at [97.40, 97.60], then the zero would be ~97.5,
# which is less than 99.557. So the sorted list should contain it... 
# Unless I made an error. Let me check.
print("\nLast 5 refined zeros (unsorted order):")
for i in range(75, 80):
 print(f" zeros[{i}] = {float(zeros_sorted[i]):.4f}")

print("\nThe sign change at index 76 (77th detected):")
a, b = sign_changes[76][0], sign_changes[76][1]
print(f" [{float(a):.2f}, {float(b):.2f}]")

# Actually wait - I already sorted. zeros_sorted has the correct 80 smallest.
# Let me just count them directly:
# All 80 sorted zeros end at gamma_80 = 99.557
# But sign changes 80-85 (indices 80-85) give zeros at 97.5, 97.7, 97.9, 98.1, 98.7, 99.1
# These are ALL less than 99.557! So we're MISSING at least 5 zeros!

# The issue is: we only refined the first 80 sign change intervals, 
# but the indices 80-84 also yield zeros smaller than our current gamma_80.
# This means our list has the WRONG zeros!

# What happened: our "first 80 sign changes" includes some at high t 
# (like index 76 at [99.40, 99.60]) that should be replaced by zeros from 
# indices 80-84 which are at lower t.

# Wait no, the sign changes are detected in ORDER. Let me re-examine.
# The scan goes t = 8.0, 8.2, 8.4, ... 
# Sign changes are recorded when Z changes sign between consecutive t values.
# So sign_change[0] is the first sign change at the lowest t.
# sign_change[79] should be at the 80th lowest sign change location.
# And sign_change[80] should be at a HIGHER t than sign_change[79].

# But that means sign_change 80 at [97.60, 97.80] has a HIGHER lower bound
# than sign_change 79 at [96.80, 97.00]. So the zero from sc[80] is ~97.7,
# while the zero from sc[79] is ~96.9. Both are less than gamma_80 = 99.56.
# 
# But our list only refined 80 zeros (indices 0-79). Index 79 gives a zero ~96.9 
# which is in our sorted list. The zeros from indices 80-84 are NOT in our list.
# They would give additional zeros at ~97.5, ~97.7, ~97.9, ~98.1, ~98.7.
# These are all SMALLER than our current max = 99.557.
# So we have the wrong 80th zero!

# The fix: we need to refine MORE sign changes and then sort and take the 80 smallest.
# We have 85 sign changes below t=100, so let's refine all 85 and take the smallest 80.
print("\nNeed to refine additional sign changes to get correct 80 smallest zeros!")
print(f"We have {len(sign_changes)} total sign changes. Need to refine at least 85.")


Sign change 76: [96.00, 96.20]
Sign change 77: [96.20, 96.40]
Sign change 78: [96.60, 96.80]
Sign change 79: [96.80, 97.00]
Sign change 80: [97.40, 97.60]

Last 5 refined zeros (unsorted order):
 zeros[75] = 95.3201
 zeros[76] = 97.1910
 zeros[77] = 97.2183
 zeros[78] = 97.2938
 zeros[79] = 99.5572

The sign change at index 76 (77th detected):
 [96.20, 96.40]

Need to refine additional sign changes to get correct 80 smallest zeros!
We have 233 total sign changes. Need to refine at least 85.


In [34]:
# Wait, I think I'm confused. Let me re-examine. The sign_changes list is in detection order.
# sign_changes[76] is at [96.20, 96.40]. Let me check what value we refined for index 76.
# 
# We refined sign_changes[0:80], which gave zeros_list[0:80] in DETECTION order (not sorted).
# Then we sorted that list to get zeros_sorted.
# 
# The unsorted list at index 76 should correspond to sign_change[76] at [96.20, 96.40].
# But we printed zeros_sorted[76] which is the 77th SMALLEST zero, not the zero from 
# the 77th sign change.
#
# Let me re-check: the zero we computed from sign_changes[76] ([96.20, 96.40]) would be ~96.3.
# The zero from sign_changes[77] ([96.60, 96.80]) would be ~96.7.
# etc.
# 
# After sorting, these should appear in the correct positions. 
# Our sorted gamma_80 = 99.557, which came from sign_changes[?].
# The highest zero in our detection order should come from the sign change interval 
# that yields the largest zero.
#
# I think the issue is that we DO have 80 zeros from the first 80 sign changes,
# and they are the 80 zeros with the smallest t-values. The sign changes 80-84 
# have LARGER t-values, so they correspond to zeros we don't need.
#
# Wait but sign_change[80] is at [97.40, 97.60], and gamma_80 (sorted) = 99.557.
# If the zero from sign_change[80] is at ~97.5, it would be SMALLER than 99.557,
# meaning it should be included in the 80 smallest.
#
# Unless... one of our first 80 sign changes produces a zero > 97.5.
# Looking at our sorted list, gamma_80 = 99.557, gamma_79 = 97.294.
# So which sign change produced the 99.557 zero?

# Let me find it:
# Before sorting, the original zeros_list had the zeros in detection order.
# We overwrote zeros_list with sorted. Let me reconstruct.
# Actually I still have the sign_changes list.

# Let me look at what interval would produce gamma ≈ 99.557
# That would be around t ≈ 99.4-99.6
for i, (a, b, za, zb) in enumerate(sign_changes):
 if abs(float(a) - 99.4) < 0.3:
 print(f"Sign change {i+1}: [{float(a):.2f}, {float(b):.2f}]")

# So the sign change near 99.5 is at index...


In [35]:
# No output - let me search more broadly
for i, (a, b, za, zb) in enumerate(sign_changes[:90]):
 if float(a) > 99.0:
 print(f"Sign change {i+1}: [{float(a):.2f}, {float(b):.2f}]")


Sign change 86: [100.40, 100.60]
Sign change 87: [100.60, 100.80]
Sign change 88: [100.80, 101.00]
Sign change 89: [101.00, 101.20]
Sign change 90: [101.80, 102.00]


In [36]:
# Hmm, no sign change near 99.5 in the first 90 entries!
# Let me look at all sign changes with a < 100:
sc_below_100 = [(i, float(a), float(b)) for i, (a, b, _, _) in enumerate(sign_changes) if float(a) < 100]
print(f"Sign changes with a < 100: {len(sc_below_100)}")
for idx, a, b in sc_below_100[-10:]:
 print(f" Index {idx}: [{a:.2f}, {b:.2f}]")


Sign changes with a < 100: 85
 Index 75: [96.00, 96.20]
 Index 76: [96.20, 96.40]
 Index 77: [96.60, 96.80]
 Index 78: [96.80, 97.00]
 Index 79: [97.40, 97.60]
 Index 80: [97.60, 97.80]
 Index 81: [97.80, 98.00]
 Index 82: [98.00, 98.20]
 Index 83: [98.60, 98.80]
 Index 84: [99.00, 99.20]


In [37]:
# So there are exactly 85 sign changes with a < 100. The sign change nearest to 99.5 is 
# at index 84: [99.00, 99.20]. No sign change has a ≈ 99.4 or 99.5!
# 
# So where did gamma_80 = 99.557 come from in our sorted list?
# 
# Let me check: was there a sign change in the original first 80 that is actually 
# near t=99.5? Let me look at all first 80 sign change intervals:

for i in range(75, 80):
 a, b = sign_changes[i][0], sign_changes[i][1]
 print(f"Sign change {i}: [{float(a):.2f}, {float(b):.2f}]")

# Wait - sign_changes[76] = [96.20, 96.40], not anywhere near 99.5.
# But in our sorted zeros, gamma_80 = 99.557. That zero must have come from 
# refining one of the first 80 sign changes. Let me find it by finding which
# sign change interval could possibly yield a zero at ~99.6.

# Actually, I bet the findroot solver, given a starting interval like [96.20, 96.40]
# with Illinois method, might have JUMPED to a different zero at 99.5! The Illinois
# method is a modified false position method and can converge to a root outside the 
# initial bracket! 

# Let me check: which of our refined zeros are OUTSIDE their original bracket?
# I need to redo the refinement and track.

# Actually, wait. Looking more carefully: 
# sign_change 76 is [99.40, 99.60] according to the earlier print... no.
# Let me re-examine - I had a confusing output earlier.

# Earlier I printed:
# Sign change 76: [96.00, 96.20]
# Sign change 77: [96.20, 96.40]
# Sign change 78: [96.60, 96.80]
# Sign change 79: [96.80, 97.00]
# Sign change 80: [97.40, 97.60]
#
# These are the detection-order indices 75-79 (0-based), which are the 76th-80th sign changes.
# None of them are near 99.5. So gamma_80=99.557 should NOT be in our list.
# 
# Unless the Illinois solver converged to a zero far from the initial bracket.
# Let me verify by recomputing.

mpmath.mp.dps = 50

# Recompute the last few zeros and check
for i in range(75, 80):
 a_val, b_val = float(sign_changes[i][0]), float(sign_changes[i][1])
 zero = mpmath.findroot(Z_for_root, (a_val, b_val), solver='illinois', tol=mpmath.mpf(10)**(-45))
 print(f" SC[{i}]: [{a_val:.2f}, {b_val:.2f}] -> zero = {float(zero):.6f}")


Sign change 75: [96.00, 96.20]
Sign change 76: [96.20, 96.40]
Sign change 77: [96.60, 96.80]
Sign change 78: [96.80, 97.00]
Sign change 79: [97.40, 97.60]


 SC[75]: [96.00, 96.20] -> zero = 94.943529


 SC[76]: [96.20, 96.40] -> zero = 99.557150


 SC[77]: [96.60, 96.80] -> zero = 97.191006


 SC[78]: [96.80, 97.00] -> zero = 97.218288


 SC[79]: [97.40, 97.60] -> zero = 97.293797


In [38]:
# AH HA! The Illinois solver is jumping to zeros far from the bracket!
# SC[75]: bracket [96.00, 96.20] -> zero at 94.94 (outside bracket!) 
# SC[76]: bracket [96.20, 96.40] -> zero at 99.56 (WAY outside bracket!)
# SC[77]: bracket [96.60, 96.80] -> zero at 97.19 (outside bracket!)
#
# This is a serious problem. The Illinois/secant method doesn't guarantee 
# staying within the bracket. We need to use bisection instead, which is 
# guaranteed to stay within [a, b].
#
# Let me implement proper bisection:

def bisect_zero(a, b, tau_coeffs, N_terms=50, max_iter=170):
 """Find zero of Z(t) in [a, b] using pure bisection."""
 za = Z(a, tau_coeffs, N_terms)
 zb = Z(b, tau_coeffs, N_terms)
 
 if za * zb > 0:
 return None # No sign change
 
 for _ in range(max_iter):
 mid = (a + b) / 2
 zm = Z(mid, tau_coeffs, N_terms)
 
 if zm == 0:
 return mid
 
 if za * zm < 0:
 b = mid
 zb = zm
 else:
 a = mid
 za = zm
 
 return (a + b) / 2

# Test: bisect the problematic intervals
mpmath.mp.dps = 50

for i in [75, 76, 77, 78, 79]:
 a, b = sign_changes[i][0], sign_changes[i][1]
 zero = bisect_zero(a, b, tau, N_terms=50, max_iter=170)
 if zero is not None:
 print(f"SC[{i}]: [{float(a):.2f}, {float(b):.2f}] -> {mpmath.nstr(zero, 40)}")
 else:
 print(f"SC[{i}]: [{float(a):.2f}, {float(b):.2f}] -> No zero (no sign change)")


SC[75]: [96.00, 96.20] -> 96.00429883228362158437843750130784498144


SC[76]: [96.20, 96.40] -> No zero (no sign change)


SC[77]: [96.60, 96.80] -> No zero (no sign change)


SC[78]: [96.80, 97.00] -> No zero (no sign change)


SC[79]: [97.40, 97.60] -> 97.39999999999999999999999999999999999994


In [39]:
# Interesting! SC[76], SC[77], SC[78] have NO sign change when checked with bisection.
# SC[79] converges to the endpoint 97.4, which is suspicious.
# SC[75] gives a zero at 96.004.
#
# The issue is that Z(t) at these extreme heights has VERY small values (like 10^{-60}),
# and the sign detection might have been unreliable due to numerical precision issues
# at dps=40 (which was used during scanning).
#
# Let me re-scan more carefully at higher precision for t > 90:

mpmath.mp.dps = 60

print("Re-scanning Z(t) for t in [90, 105] at dps=60, step=0.1:")
sign_changes_v2 = []
t = mpmath.mpf('90.0')
t_end = mpmath.mpf('105.0')
step = mpmath.mpf('0.1')

z_prev = Z(t, tau, 50)
t_prev = t

while t <= t_end:
 t = t + step
 z_curr = Z(t, tau, 50)
 
 if z_prev * z_curr < 0:
 sign_changes_v2.append((t_prev, t, z_prev, z_curr))
 print(f" Sign change: [{float(t_prev):.1f}, {float(t):.1f}] Z={float(z_prev):+.3e} -> {float(z_curr):+.3e}")
 
 z_prev = z_curr
 t_prev = t

print(f"\nFound {len(sign_changes_v2)} sign changes in [90, 105]")


Re-scanning Z(t) for t in [90, 105] at dps=60, step=0.1:


 Sign change: [90.4, 90.5] Z=+8.541e-57 -> -6.375e-57


 Sign change: [91.1, 91.2] Z=-7.111e-58 -> +4.604e-57


 Sign change: [92.4, 92.5] Z=+6.455e-58 -> -7.515e-58


 Sign change: [93.7, 93.8] Z=-1.571e-58 -> +5.995e-59


...

 Sign change: [100.2, 100.3] Z=-8.225e-64 -> +2.542e-63


 Sign change: [101.3, 101.4] Z=+8.803e-64 -> -1.070e-63


 Sign change: [103.1, 103.2] Z=-8.957e-65 -> +3.148e-65


 Sign change: [103.8, 103.9] Z=+7.228e-66 -> -3.018e-65



Found 13 sign changes in [90, 105]


In [40]:
# At dps=60 with step=0.1, we find far fewer sign changes in [90,105] - only 13.
# This is much more reasonable. The earlier scan at dps=40 was producing spurious 
# sign changes due to precision loss.
#
# The values of Z(t) are extremely small (10^{-57} to 10^{-65}), and at dps=40 
# (about 40 decimal digits), the relative precision is barely enough to determine 
# the sign correctly.
#
# Let me redo the ENTIRE scan at dps=60 with step=0.1 for the region where 
# precision matters (say t > 50). For t < 50, the values are larger and dps=40 was fine.

# First, let's count how many zeros the original scan found reliably (t < 50):
reliable_sc = [(i, float(a), float(b)) for i, (a, b, _, _) in enumerate(sign_changes) if float(a) < 50]
print(f"Sign changes with a < 50 from original scan: {len(reliable_sc)}")

# Let me redo the scan for the whole range at dps=60
mpmath.mp.dps = 60

print("\nRe-scanning Z(t) for t in [8, 120] at dps=60, step=0.1...")
import time
t_scan_start = time.time()

all_sign_changes = []
t = mpmath.mpf('8.0')
t_end = mpmath.mpf('120.0')
step = mpmath.mpf('0.1')

z_prev = Z(t, tau, 50)
t_prev = t

count = 0
while t <= t_end:
 t = t + step
 z_curr = Z(t, tau, 50)
 
 if z_prev * z_curr < 0:
 all_sign_changes.append((t_prev, t))
 
 z_prev = z_curr
 t_prev = t
 count += 1
 
 if count % 200 == 0:
 elapsed = time.time() - t_scan_start
 print(f" t={float(t):.1f}, found {len(all_sign_changes)} sign changes, elapsed: {elapsed:.1f}s")

elapsed = time.time() - t_scan_start
print(f"\nScan complete: {len(all_sign_changes)} sign changes found in {elapsed:.1f}s")


Sign changes with a < 50 from original scan: 20

Re-scanning Z(t) for t in [8, 120] at dps=60, step=0.1...


 t=28.0, found 7 sign changes, elapsed: 55.8s


 t=48.0, found 18 sign changes, elapsed: 122.2s


 t=68.0, found 33 sign changes, elapsed: 196.7s


 t=88.0, found 48 sign changes, elapsed: 279.2s


 t=108.0, found 69 sign changes, elapsed: 370.6s



Scan complete: 123 sign changes found in 422.6s


In [41]:
# We found only 123 sign changes up to t=120 at proper precision. 
# We need 80 zeros. Let's check how many are below various thresholds:
below_100 = sum(1 for a, b in all_sign_changes if float(a) < 100)
below_110 = sum(1 for a, b in all_sign_changes if float(a) < 110)
below_120 = sum(1 for a, b in all_sign_changes if float(a) < 120)
print(f"Sign changes below 100: {below_100}")
print(f"Sign changes below 110: {below_110}")
print(f"Sign changes below 120: {below_120}")

# Good - we have way more than 80. Now refine the first 85 using bisection 
# (to have some margin), then sort and take 80.
# But bisection at dps=60 is slow. Let me use fewer iterations (100 gives ~30 digits from 0.1 interval)
# 100 bisection steps: 0.1 * 2^{-100} ≈ 10^{-31} precision. That's 31 digits.
# For 35+ digits: 120 steps gives ~36 digits. For 40 digits: 135 steps.

mpmath.mp.dps = 55
import time

# Need at least 80 sign changes. Let's refine the first 85.
N_refine = min(85, len(all_sign_changes))
print(f"\nRefining {N_refine} zeros using bisection (135 iterations each)...")
t0 = time.time()

refined_zeros = []
for i in range(N_refine):
 a, b = all_sign_changes[i]
 zero = bisect_zero(a, b, tau, N_terms=50, max_iter=135)
 if zero is not None:
 refined_zeros.append(zero)
 
 if (i+1) % 10 == 0:
 elapsed = time.time() - t0
 print(f" {i+1}/{N_refine} done, elapsed: {elapsed:.1f}s, zeros found: {len(refined_zeros)}")

total = time.time() - t0
print(f"\nRefined {len(refined_zeros)} zeros in {total:.1f}s")


TimeoutError: Code execution timed out after 339.0 seconds